In [0]:
%run ../../utils/utils

## Gold — Volumetria das Tabelas (Bronze / Silver / Quarentena / Falhas)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType
from datetime import datetime, timezone

## Parâmetros

In [0]:
TABELAS_PROJETO = [
    "ecommerce_clientes",
    "ecommerce_pedidos",
    "ecommerce_enderecos",
    "ecommerce_itens_pedido",
    "ecommerce_rastreamento",
    "ecommerce_produtos",
    "ecommerce_categorias"
]

In [0]:
IDS_POR_TABELA = {
    "ecommerce_clientes": "id_cliente",
    "ecommerce_pedidos": "id_pedido",
    "ecommerce_enderecos": "id_endereco",
    "ecommerce_itens_pedido": "id_item_pedido",
    "ecommerce_rastreamento": "id_rastreamento",
    "ecommerce_produtos": "sku",
    "ecommerce_categorias": "id_categoria"
}
 
NOME_TABELA_GOLD = "gold_volumetria_tabelas"
TIMESTAMP_EXECUCAO = datetime.now(timezone.utc)
 
SCHEMA_GOLD_VOLUMETRIA = StructType([
    StructField("nome_tabela_origem", StringType(), True),
    StructField("total_registros_bronze", LongType(), True),
    StructField("total_registros_bronze_distintos", LongType(), True),
    StructField("total_registros_silver", LongType(), True),
    StructField("total_registros_quarentena", LongType(), True),
    StructField("total_eventos_dq_logs", LongType(), True),
    StructField("total_falhas_registros", LongType(), True),
    StructField("percentual_falha", DoubleType(), True),
    StructField("data_verificacao", TimestampType(), True),
])
 
print(f"===== INICIANDO VOLUMETRIA GOLD PARA {len(TABELAS_PROJETO)} TABELAS =====")
print("Tabela Gold de destino:", NOME_TABELA_GOLD)
print(
    "\nATENÇÃO: dq_monitoring_logs é cumulativo (append-only) desde sempre. "
    "O percentual de falha abaixo reflete TODO o histórico de execuções já "
    "gravado, não só o estado atual da Bronze."
)
 

## Contagem por camada (Bronze / Silver / Quarentena / Falhas)

In [0]:
if delta_existe("", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_dq_logs = ler_delta("", "dq_monitoring_logs", STORAGE_OPTIONS)
    dq_logs_disponivel = True
    print(f"dq_monitoring_logs encontrada — total geral de linhas de log: {df_dq_logs.count()}")
else:
    df_dq_logs = None
    dq_logs_disponivel = False
    print("Aviso: tabela dq_monitoring_logs não encontrada na raiz do Data Lake.")
 
registros_volumetria = []
 
for tabela in TABELAS_PROJETO:
    qtd_bronze = contar_delta_seguro("bronze", tabela)
    qtd_bronze_distintos = contar_distintos_bronze_seguro(tabela)
    qtd_silver = contar_delta_seguro("silver", tabela)
    qtd_quarentena = contar_delta_seguro("silver/quarentena", tabela)
 
    if dq_logs_disponivel:
        df_logs_tabela = df_dq_logs.filter(F.col("tabela") == tabela)
        qtd_eventos_dq_logs = df_logs_tabela.count()
        total_falhas_registros = df_logs_tabela.agg(
            F.sum("qtd_registros_falhos")
        ).collect()[0][0] or 0
    else:
        qtd_eventos_dq_logs = 0
        total_falhas_registros = 0
 
    if qtd_bronze_distintos > 0:
        percentual_falha = round((total_falhas_registros / qtd_bronze_distintos) * 100, 2)
    else:
        percentual_falha = None
 
    registros_volumetria.append((
        tabela,
        qtd_bronze,
        qtd_bronze_distintos,
        qtd_silver,
        qtd_quarentena,
        qtd_eventos_dq_logs,
        int(total_falhas_registros),
        percentual_falha,
        TIMESTAMP_EXECUCAO,
    ))
 
    print(
        f"{tabela:30s} | bronze={qtd_bronze:>8} | bronze_distintos={qtd_bronze_distintos:>8} "
        f"| silver={qtd_silver:>8} | quarentena={qtd_quarentena:>8} "
        f"| eventos_log={qtd_eventos_dq_logs:>8} | falhas={total_falhas_registros:>8} "
        f"| perc_falha={percentual_falha}"
    )
 
df_gold_volumetria = spark.createDataFrame(registros_volumetria, schema=SCHEMA_GOLD_VOLUMETRIA)
 
print(f"\nTotal de linhas geradas: {df_gold_volumetria.count()}")
display(df_gold_volumetria)

## Gravação no Delta Lake (Gold)

In [0]:
sucesso_delta = gravar_delta(
    df=df_gold_volumetria,
    camada="gold",
    tabela=NOME_TABELA_GOLD,
    storage_opts=STORAGE_OPTIONS,
    mode="overwrite",
    particionar=False,
)
 
if sucesso_delta:
    print(f"[Sucesso] {NOME_TABELA_GOLD} gravada em Delta (pasta gold).")
else:
    print(f"[Erro] Falha ao gravar {NOME_TABELA_GOLD} em Delta.")

## Réplica no SQL Server

In [0]:
 
if sucesso_delta:
    try:
        escrever_sqlserver_gold(
            df_spark=df_gold_volumetria,
            schema="squad1",
            tabela=NOME_TABELA_GOLD,
            modo="overwrite",
        )
        print(f"[Sucesso] {NOME_TABELA_GOLD} sincronizada no SQL Server.")
    except Exception as e:
        print(f"[Erro SQL Server] Falha ao enviar {NOME_TABELA_GOLD}: {e}")
else:
    print("[Aviso] Envio ao SQL Server ignorado: a gravação Delta falhou.")
 

## Validação final

In [0]:
print("===== VALIDAÇÃO FINAL =====")
 
if delta_existe("gold", NOME_TABELA_GOLD, STORAGE_OPTIONS):
    df_validacao = ler_delta("gold", NOME_TABELA_GOLD, STORAGE_OPTIONS)
    print(f"Registros na tabela Gold {NOME_TABELA_GOLD}: {df_validacao.count()}")
    display(df_validacao)
else:
    print(f"Atenção: tabela {NOME_TABELA_GOLD} não encontrada na validação.")
 
print("===== PROCESSO CONCLUÍDO =====")